In [ ]:
!pip install -q pandas streamlit plotly requests matplotlib fpdf2 transformers sentencepiece sacremoses arabic-reshaper python-bidi pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.0/337.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 64.1 MB/s eta 0:00:00


In [ ]:
import os, getpass

os.environ["EMAIL_ADDRESS"] = "worofyousef@gmail.com"
os.environ["EMAIL_APP_PASSWORD"] = getpass.getpass("Gmail App Password: ")
os.environ["RECIPIENT_EMAIL"] = "worofyousef@gmail.com"
os.environ["NGROK_AUTH_TOKEN"] = getpass.getpass("ngrok auth token: ")

Gmail App Password: ··········
ngrok auth token: ··········


In [ ]:
!mkdir -p agents reports

In [ ]:
%%writefile config.py
import os

EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS", "worofyousef@gmail.com")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD", "")
RECIPIENT_EMAIL = os.getenv("RECIPIENT_EMAIL", "worofyousef@gmail.com")
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587

DEFAULT_CSV_PATH = "/content/gta_v_worldwide_sales_player_analytics_2013_2026.csv"
DASHBOARD_STATE_PATH = "dashboard_state.json"
REPORTS_DIR = "reports"

LLM_MODEL_NAME = "google/flan-t5-base"
WIKI_SUMMARY_URL = "https://en.wikipedia.org/api/rest_v1/page/summary/Grand_Theft_Auto_V"

ANOMALY_ZSCORE_THRESHOLD = 2.0
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 2

PREPARED_FOR = "Marketing Team Lead"

# Persistent "learned" column-role mapping, keyed by each CSV's column signature.
COLUMN_MAP_CACHE_PATH = "column_map_cache.json"

REPORT_TYPES = {
    "overall": {"title": "Overall Performance Report", "title_ar": "تقرير الأداء العام",
                "email_subject": "Overall Performance Report — For the Marketing Team Lead"},
    "regional": {"title": "Regional Deep-Dive Report", "title_ar": "تقرير تفصيلي للأداء الإقليمي",
                 "email_subject": "Regional Deep-Dive Report — For the Marketing Team Lead"},
    "platform": {"title": "Platform Performance Report", "title_ar": "تقرير أداء المنصات",
                 "email_subject": "Platform Performance Report — For the Marketing Team Lead"},
    "engagement": {"title": "Player Engagement Report", "title_ar": "تقرير تفاعل اللاعبين",
                   "email_subject": "Player Engagement Report — For the Marketing Team Lead"},
    "risk": {"title": "Anomaly & Risk Report", "title_ar": "تقرير الشذوذ والمخاطر",
             "email_subject": "Anomaly & Risk Report — For the Marketing Team Lead"},
}

Writing config.py


In [ ]:
%%writefile agents/__init__.py

Writing agents/__init__.py


In [ ]:
%%writefile agents/local_llm.py
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


class LocalLLM:
    """Loaded once, shared by SchemaAgent (column classification) and AnalysisAgent (AI commentary)."""

    def __init__(self, model_name: str):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def generate(self, prompt: str, max_new_tokens: int = 100) -> str:
        try:
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
            output_ids = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            return self.tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
        except Exception:
            return ""

Writing agents/local_llm.py


In [ ]:
%%writefile agents/schema_agent.py
import hashlib
import json
import os
import pandas as pd


class SchemaAgent:
    """Classifies each CSV column's role using the local LLM (semantic, not keyword-matching),
    validates the guess against hard dtype/uniqueness rules, and caches the result per dataset
    schema so it 'remembers' the mapping across runs. A manual correction from the UI overwrites
    the cache permanently for that schema — this is the learning/feedback loop."""

    ROLES = ["country", "platform", "sales", "players", "date", "other"]

    def __init__(self, llm, cache_path: str):
        self.llm = llm
        self.cache_path = cache_path

    def _fingerprint(self, df: pd.DataFrame) -> str:
        return hashlib.md5(",".join(sorted(df.columns)).encode()).hexdigest()

    def _load_cache(self) -> dict:
        if os.path.exists(self.cache_path):
            try:
                with open(self.cache_path) as f:
                    return json.load(f)
            except Exception:
                return {}
        return {}

    def _save_cache(self, cache: dict):
        with open(self.cache_path, "w", encoding="utf-8") as f:
            json.dump(cache, f, indent=2)

    def _profile_column(self, series: pd.Series) -> dict:
        sample = series.dropna().astype(str).head(5).tolist()
        return {
            "is_numeric": bool(pd.api.types.is_numeric_dtype(series)),
            "n_unique": int(series.nunique(dropna=True)),
            "n_rows": int(len(series)),
            "sample_values": sample,
        }

    def _classify_column_llm(self, col_name: str, profile: dict) -> str:
        prompt = (
            "You are classifying a column of a video-game worldwide sales dataset. "
            f"Column name: '{col_name}'. Sample values: {profile['sample_values']}. "
            f"Data type: {'numeric' if profile['is_numeric'] else 'text'}. "
            "Which single category best fits this column? Choose exactly one word from: "
            "country, platform, sales, players, date, other. Answer with only that one word."
        )
        answer = self.llm.generate(prompt, max_new_tokens=6).lower()
        for role in self.ROLES:
            if role in answer:
                return role
        return "other"

    def _validate_role(self, role: str, profile: dict) -> str:
        """Rule-based safety net — an LLM guess can never override these hard constraints,
        which is what stops a numeric region-code column from being accepted as 'country'."""
        if role in ("sales", "players") and not profile["is_numeric"]:
            return "other"
        if role in ("country", "platform"):
            if profile["is_numeric"]:
                return "other"
            uniqueness_ratio = profile["n_unique"] / max(profile["n_rows"], 1)
            avg_len = (sum(len(s) for s in profile["sample_values"]) / max(len(profile["sample_values"]), 1)) if profile["sample_values"] else 999
            if uniqueness_ratio > 0.5 or avg_len > 40:
                return "other"
        return role

    def profile_all(self, df: pd.DataFrame) -> dict:
        """Exposed so the UI can show sample values per column when letting the user correct a mapping."""
        return {col: self._profile_column(df[col]) for col in df.columns}

    def detect_columns(self, df: pd.DataFrame, manual_overrides: dict = None) -> dict:
        fingerprint = self._fingerprint(df)
        cache = self._load_cache()

        if manual_overrides is not None:
            cleaned = {k: v for k, v in manual_overrides.items() if v and v != "(none)"}
            cache[fingerprint] = cleaned
            self._save_cache(cache)
            return cleaned

        if fingerprint in cache:
            return cache[fingerprint]

        role_to_col = {}
        for col in df.columns:
            profile = self._profile_column(df[col])
            role = self._classify_column_llm(col, profile)
            role = self._validate_role(role, profile)
            if role != "other" and role not in role_to_col:
                role_to_col[role] = col

        cache[fingerprint] = role_to_col
        self._save_cache(cache)
        return role_to_col

Writing agents/schema_agent.py


In [ ]:
%%writefile agents/retrieval_agent.py
import pandas as pd
import requests

import config


class RetrievalAgent:
    def __init__(self, schema_agent):
        self.schema_agent = schema_agent

    def load_sales_data(self, csv_path: str, manual_overrides: dict = None) -> dict:
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]
        columns = self.schema_agent.detect_columns(df, manual_overrides=manual_overrides)
        return {"df": df, "columns": columns}

    def fetch_exchange_rate(self, base="USD", target="EGP"):
        try:
            resp = requests.get(f"https://api.exchangerate-api.com/v4/latest/{base}", timeout=8)
            resp.raise_for_status()
            return resp.json()["rates"].get(target)
        except Exception:
            return None

    def fetch_context_summary(self):
        try:
            resp = requests.get(config.WIKI_SUMMARY_URL, timeout=8)
            resp.raise_for_status()
            return resp.json().get("extract")
        except Exception:
            return None

    def retrieve_all(self, csv_path: str, manual_overrides: dict = None) -> dict:
        sales = self.load_sales_data(csv_path, manual_overrides=manual_overrides)
        fx_rate = self.fetch_exchange_rate()
        context_summary = self.fetch_context_summary()
        return {
            "sales_df": sales["df"],
            "columns": sales["columns"],
            "usd_to_egp": fx_rate,
            "context_summary": context_summary,
        }

Writing agents/retrieval_agent.py


In [ ]:
%%writefile agents/analysis_agent.py
import pandas as pd


class AnalysisAgent:
    def __init__(self, llm):
        self.llm = llm

    # ---------- Stats ----------
    def compute_stats(self, df: pd.DataFrame, columns: dict) -> dict:
        stats = {"rows": len(df)}
        sales_col = columns.get("sales")
        country_col = columns.get("country")
        platform_col = columns.get("platform")
        players_col = columns.get("players")

        if sales_col:
            sales_numeric = pd.to_numeric(df[sales_col], errors="coerce")
            stats["total_sales"] = float(sales_numeric.sum(skipna=True))
            stats["avg_sales"] = float(sales_numeric.mean(skipna=True))
        if country_col and sales_col:
            grouped = df.assign(_sales=pd.to_numeric(df[sales_col], errors="coerce"))
            stats["top_countries"] = (
                grouped.groupby(country_col)["_sales"].sum().sort_values(ascending=False).head(5).to_dict()
            )
        if platform_col and sales_col:
            grouped = df.assign(_sales=pd.to_numeric(df[sales_col], errors="coerce"))
            stats["top_platforms"] = (
                grouped.groupby(platform_col)["_sales"].sum().sort_values(ascending=False).head(5).to_dict()
            )
        if players_col:
            players_numeric = pd.to_numeric(df[players_col], errors="coerce")
            stats["total_players"] = float(players_numeric.sum(skipna=True))
            stats["avg_players"] = float(players_numeric.mean(skipna=True))
        return stats

    def get_sales_series(self, df: pd.DataFrame, columns: dict):
        sales_col = columns.get("sales")
        if not sales_col:
            return None
        return pd.to_numeric(df[sales_col], errors="coerce").dropna().tolist()

    # ---------- Anomalies ----------
    def detect_anomalies(self, df: pd.DataFrame, columns: dict, z_threshold: float = 2.0) -> list:
        anomalies = []
        sales_col = columns.get("sales")
        if not sales_col:
            return anomalies
        sales_numeric = pd.to_numeric(df[sales_col], errors="coerce")

        for group_field in ("country", "platform"):
            col = columns.get(group_field)
            if not col:
                continue
            grouped = df.assign(_sales=sales_numeric).groupby(col)["_sales"].sum().dropna()
            if len(grouped) < 3:
                continue
            mean, std = grouped.mean(), grouped.std()
            if not std or pd.isna(std) or std == 0:
                continue
            z_scores = (grouped - mean) / std
            outliers = z_scores[z_scores.abs() >= z_threshold]
            for name, z in outliers.items():
                anomalies.append({
                    "group": group_field, "name": str(name),
                    "value": float(grouped[name]), "z_score": round(float(z), 2),
                })
        return anomalies[:5]

    # ---------- Recommendations ----------
    def generate_recommendations(self, report_type: str, stats: dict, anomalies: list) -> list:
        recs = []
        top_country = next(iter(stats["top_countries"]), None) if stats.get("top_countries") else None
        top_platform = next(iter(stats["top_platforms"]), None) if stats.get("top_platforms") else None

        if report_type == "regional":
            if top_country:
                recs.append(f"Allocate additional regional marketing budget to {top_country}, the current leading market.")
            for country, val in list(stats.get("top_countries", {}).items())[1:3]:
                recs.append(f"Consider a growth campaign in {country} (current sales: {val:,.2f}) to close the gap with the top market.")
        elif report_type == "platform":
            if top_platform:
                recs.append(f"Prioritize {top_platform}-specific promotions and store placement, the current best-performing platform.")
            for plat, val in list(stats.get("top_platforms", {}).items())[1:3]:
                recs.append(f"Evaluate platform-specific bundles for {plat} (current sales: {val:,.2f}).")
        elif report_type == "engagement":
            if "avg_players" in stats:
                recs.append(f"Average engagement is {stats['avg_players']:,.0f} players per entry — set up retention campaigns if this trend declines.")
            recs.append("Consider in-game events or seasonal content to sustain player engagement across regions.")
        elif report_type == "risk":
            if anomalies:
                for a in anomalies:
                    recs.append(f"Investigate the anomaly in {a['name']} ({a['group']}, z-score {a['z_score']}) before the next budget cycle.")
            else:
                recs.append("No statistically significant anomalies detected this run — no immediate risk action needed.")
        else:
            if top_country:
                recs.append(f"Increase marketing investment in {top_country}, the leading market by sales.")
            if top_platform:
                recs.append(f"Prioritize platform-specific content and optimization for {top_platform}.")
            for a in anomalies[:2]:
                recs.append(f"Investigate the unusual sales pattern in {a['name']} ({a['group']}) — z-score {a['z_score']}.")
            if "avg_players" in stats:
                recs.append(f"Average engagement is {stats['avg_players']:,.0f} players per entry — monitor this trend closely.")

        if not recs:
            recs.append("Not enough detected columns to generate specific recommendations — verify column mapping.")
        return recs

    # ---------- Deterministic bilingual core (numbers are Python-formatted, never machine-translated) ----------
    def _core_numbers(self, stats: dict, fx_rate):
        rows = stats.get("rows", 0)
        total_sales = stats.get("total_sales")
        egp = (total_sales * fx_rate) if (total_sales and fx_rate) else None
        top_country = next(iter(stats.get("top_countries", {})), None)
        top_platform = next(iter(stats.get("top_platforms", {})), None)
        avg_players = stats.get("avg_players")
        total_players = stats.get("total_players")
        return rows, total_sales, egp, top_country, top_platform, avg_players, total_players

    def build_bilingual_summary(self, report_type: str, stats: dict, fx_rate, anomalies: list) -> dict:
        rows, total_sales, egp, top_country, top_platform, avg_players, total_players = self._core_numbers(stats, fx_rate)
        egp_en = f" (~{egp:,.0f} EGP)" if egp else ""
        egp_ar = f" (حوالي {egp:,.0f} جنيه مصري)" if egp else ""

        if report_type == "regional":
            countries = stats.get("top_countries", {})
            lines = [f"{c}: {v:,.2f}" for c, v in countries.items()]
            en = (f"This regional deep-dive covers {rows} records across {len(countries)} tracked markets. "
                  f"Top markets by sales — " + "; ".join(lines) + ". "
                  + (f"{top_country} leads all markets." if top_country else ""))
            ar = (f"يغطي هذا التقرير الإقليمي التفصيلي {rows} سجلاً عبر {len(countries)} سوقاً. "
                  f"أفضل الأسواق من حيث المبيعات — " + "؛ ".join(lines) + ". "
                  + (f"يتصدر سوق {top_country} جميع الأسواق." if top_country else ""))

        elif report_type == "platform":
            platforms = stats.get("top_platforms", {})
            lines = [f"{p}: {v:,.2f}" for p, v in platforms.items()]
            en = (f"This platform performance report covers {rows} records across {len(platforms)} tracked platforms. "
                  f"Top platforms by sales — " + "; ".join(lines) + ". "
                  + (f"{top_platform} is the best-performing platform." if top_platform else ""))
            ar = (f"يغطي تقرير أداء المنصات هذا {rows} سجلاً عبر {len(platforms)} منصة. "
                  f"أفضل المنصات من حيث المبيعات — " + "؛ ".join(lines) + ". "
                  + (f"تُعد منصة {top_platform} الأفضل أداءً." if top_platform else ""))

        elif report_type == "engagement":
            en = (f"This player engagement report covers {rows} records. "
                  + (f"Total tracked players: {total_players:,.0f}. " if total_players else "")
                  + (f"Average players per record: {avg_players:,.0f}. " if avg_players else "")
                  + (f"{top_platform} shows the strongest platform engagement by sales." if top_platform else ""))
            ar = (f"يغطي تقرير تفاعل اللاعبين هذا {rows} سجلاً. "
                  + (f"إجمالي اللاعبين المسجلين: {total_players:,.0f}. " if total_players else "")
                  + (f"متوسط عدد اللاعبين لكل سجل: {avg_players:,.0f}. " if avg_players else "")
                  + (f"تُظهر منصة {top_platform} أقوى تفاعل من حيث المبيعات." if top_platform else ""))

        elif report_type == "risk":
            if anomalies:
                lines = [f"{a['name']} ({a['group']}, z-score {a['z_score']})" for a in anomalies]
                en = f"This risk report covers {rows} records. Detected anomalies: " + "; ".join(lines) + "."
                ar = f"يغطي تقرير المخاطر هذا {rows} سجلاً. الحالات الشاذة المكتشفة: " + "؛ ".join(lines) + "."
            else:
                en = f"This risk report covers {rows} records. No statistically significant anomalies were detected this run."
                ar = f"يغطي تقرير المخاطر هذا {rows} سجلاً. لم يتم اكتشاف أي حالات شاذة ذات دلالة إحصائية في هذا التشغيل."

        else:  # overall
            en = (f"The dataset covers {rows} records. Total sales stand at {total_sales:,.2f}{egp_en}. "
                  + (f"{top_country} is the top-performing market. " if top_country else "")
                  + (f"{top_platform} leads by platform. " if top_platform else "")
                  + (f"Average player count per entry is {avg_players:,.0f}." if avg_players else ""))
            ar = (f"تغطي مجموعة البيانات {rows} سجلاً. يبلغ إجمالي المبيعات {total_sales:,.2f} دولار{egp_ar}. "
                  + (f"{top_country} هي السوق الأعلى أداءً. " if top_country else "")
                  + (f"تتصدر منصة {top_platform} من حيث المبيعات. " if top_platform else "")
                  + (f"يبلغ متوسط عدد اللاعبين لكل سجل {avg_players:,.0f} لاعب." if avg_players else ""))

        return {"en": en.strip(), "ar": ar.strip()}

    # ---------- Optional AI commentary (local LLM — qualitative only, never carries numbers) ----------
    def generate_ai_commentary(self, report_type: str, stats: dict) -> str:
        prompt = (
            f"In two sentences, give a marketing-oriented interpretive comment (no numbers) about a "
            f"video game {report_type} analytics report. Focus on strategic implications, not data restatement."
        )
        text = self.llm.generate(prompt, max_new_tokens=80)
        if len(text.split()) < 6:
            return ""
        return text

Writing agents/analysis_agent.py


In [ ]:
%%writefile agents/action_agent.py
import json
import os
import re
import datetime
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication

import requests
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from fpdf import FPDF
import arabic_reshaper
from bidi.algorithm import get_display

import config


class ActionAgent:
    def __init__(self):
        os.makedirs(config.REPORTS_DIR, exist_ok=True)
        self._arabic_font_path = self._ensure_arabic_font()

    def _ensure_arabic_font(self):
        font_path = os.path.join(config.REPORTS_DIR, "NotoNaskhArabic-Regular.ttf")
        if os.path.exists(font_path):
            return font_path
        try:
            url = "https://raw.githubusercontent.com/googlefonts/noto-fonts/main/hinted/ttf/NotoNaskhArabic/NotoNaskhArabic-Regular.ttf"
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()
            with open(font_path, "wb") as f:
                f.write(resp.content)
            return font_path
        except Exception:
            return None

    def _shape_arabic(self, text: str) -> str:
        try:
            return get_display(arabic_reshaper.reshape(text))
        except Exception:
            return text

    def _sanitize_pdf_text(self, text, max_word_len: int = 50) -> str:
        if text is None:
            return ""
        text = str(text)
        text = text.encode("latin-1", "replace").decode("latin-1")

        def _break(match):
            word = match.group(0)
            return " ".join(word[i:i + max_word_len] for i in range(0, len(word), max_word_len))

        return re.sub(r"\S{%d,}" % (max_word_len + 1), _break, text)

    def _safe_multicell(self, pdf, h, text, **kwargs):
        try:
            pdf.multi_cell(0, h, self._sanitize_pdf_text(text), **kwargs)
        except Exception:
            try:
                pdf.multi_cell(0, h, "[content omitted]")
            except Exception:
                pass

    def _bar_chart(self, data_dict: dict, title: str, path: str):
        if not data_dict:
            return None
        try:
            fig, ax = plt.subplots(figsize=(6, 3.5))
            ax.bar([str(k)[:20] for k in data_dict.keys()], list(data_dict.values()), color="#3E7CB1")
            ax.set_title(title)
            plt.xticks(rotation=30, ha="right")
            plt.tight_layout()
            fig.savefig(path)
            plt.close(fig)
            return path
        except Exception:
            return None

    def _histogram(self, values: list, title: str, path: str):
        if not values:
            return None
        try:
            fig, ax = plt.subplots(figsize=(6, 3.5))
            ax.hist(values, bins=30, color="#8E44AD")
            ax.set_title(title)
            plt.tight_layout()
            fig.savefig(path)
            plt.close(fig)
            return path
        except Exception:
            return None

    def generate_report(
        self, report_type: str, stats: dict, insights: dict, ai_commentary: str = "",
        anomalies: list = None, context_summary: str = None,
        recommendations: list = None, sales_values: list = None,
    ) -> str:
        anomalies = anomalies or []
        recommendations = recommendations or []
        meta = config.REPORT_TYPES.get(report_type, config.REPORT_TYPES["overall"])

        chart_country = None
        chart_platform = None
        chart_hist = None
        if report_type in ("overall", "regional"):
            chart_country = self._bar_chart(stats.get("top_countries"), "Top countries by sales",
                                             os.path.join(config.REPORTS_DIR, "chart_country.png"))
        if report_type in ("overall", "platform"):
            chart_platform = self._bar_chart(stats.get("top_platforms"), "Top platforms by sales",
                                              os.path.join(config.REPORTS_DIR, "chart_platform.png"))
        if report_type in ("overall", "engagement"):
            chart_hist = self._histogram(sales_values, "Sales distribution across records",
                                          os.path.join(config.REPORTS_DIR, "chart_hist.png"))

        pdf = FPDF()
        pdf.add_page()

        try:
            pdf.set_font("Helvetica", "B", 16)
            pdf.cell(0, 10, f"GTA V Analytics — {meta['title']}", ln=True)
            pdf.set_font("Helvetica", "I", 10)
            pdf.cell(0, 6, f"Prepared for: {config.PREPARED_FOR}", ln=True)
            pdf.cell(0, 6, f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}", ln=True)
            pdf.ln(3)
        except Exception:
            pass

        try:
            pdf.set_font("Helvetica", "B", 12)
            pdf.cell(0, 8, "Summary", ln=True)
            pdf.set_font("Helvetica", size=11)
            self._safe_multicell(pdf, 6, insights.get("en", ""))
            pdf.ln(2)
        except Exception:
            pass

        if ai_commentary:
            try:
                pdf.set_font("Helvetica", "B", 11)
                pdf.cell(0, 7, "AI Commentary", ln=True)
                pdf.set_font("Helvetica", "I", 10)
                self._safe_multicell(pdf, 5, ai_commentary)
                pdf.ln(2)
            except Exception:
                pass

        try:
            pdf.set_font("Helvetica", "B", 12)
            pdf.cell(0, 8, "Recommendations", ln=True)
            pdf.set_font("Helvetica", size=10)
            for r in recommendations:
                self._safe_multicell(pdf, 5, f"- {r}")
            pdf.ln(2)
        except Exception:
            pass

        if report_type == "risk" and anomalies:
            try:
                pdf.set_font("Helvetica", "B", 12)
                pdf.cell(0, 8, "Detected Anomalies", ln=True)
                pdf.set_font("Helvetica", size=10)
                for a in anomalies:
                    self._safe_multicell(pdf, 5, f"- {a.get('name')} ({a.get('group')}): z-score {a.get('z_score')}, total {a.get('value', 0):.2f}")
                pdf.ln(2)
            except Exception:
                pass

        for chart_path, caption in ((chart_country, "Top countries by sales"),
                                     (chart_platform, "Top platforms by sales"),
                                     (chart_hist, "Sales distribution")):
            if chart_path and os.path.exists(chart_path):
                try:
                    pdf.set_font("Helvetica", "I", 9)
                    pdf.cell(0, 6, caption, ln=True)
                    pdf.image(chart_path, w=170)
                    pdf.set_xy(pdf.l_margin, pdf.get_y())
                    pdf.ln(3)
                except Exception:
                    pass

        if context_summary and report_type == "overall":
            try:
                pdf.set_font("Helvetica", "B", 12)
                pdf.cell(0, 8, "Background Context", ln=True)
                pdf.set_font("Helvetica", "I", 9)
                self._safe_multicell(pdf, 5, context_summary)
                pdf.ln(2)
            except Exception:
                pass

        if insights.get("ar") and self._arabic_font_path:
            try:
                pdf.set_font("Helvetica", "B", 12)
                pdf.cell(0, 8, f"{meta['title_ar']} — الملخص", ln=True)
                pdf.add_font("NotoArabic", "", self._arabic_font_path)
                pdf.set_font("NotoArabic", size=11)
                pdf.multi_cell(0, 6, self._shape_arabic(insights["ar"]), align="R")
                pdf.ln(2)
            except Exception:
                pass

        try:
            pdf.set_font("Helvetica", "I", 8)
            self._safe_multicell(pdf, 5, f"Raw stats: {json.dumps(stats, default=str)}")
        except Exception:
            pass

        out_path = os.path.join(config.REPORTS_DIR, f"gta_v_report_{report_type}.pdf")
        pdf.output(out_path)
        return out_path

    def send_email(self, report_type: str, report_path: str, insights: dict, recipient: str = None) -> bool:
        recipient = recipient or config.RECIPIENT_EMAIL
        if not config.EMAIL_APP_PASSWORD:
            raise RuntimeError("EMAIL_APP_PASSWORD is not set.")
        meta = config.REPORT_TYPES.get(report_type, config.REPORT_TYPES["overall"])

        msg = MIMEMultipart()
        msg["From"] = config.EMAIL_ADDRESS
        msg["To"] = recipient
        msg["Subject"] = meta["email_subject"]
        body = (
            f"Hi,\n\nPlease find attached the {meta['title']} — prepared for the {config.PREPARED_FOR}.\n\n"
            f"{insights.get('en', '')}\n\n---\n{insights.get('ar', '')}\n"
        )
        msg.attach(MIMEText(body, "plain"))

        with open(report_path, "rb") as f:
            part = MIMEApplication(f.read(), _subtype="pdf")
            part.add_header("Content-Disposition", "attachment", filename=os.path.basename(report_path))
            msg.attach(part)

        with smtplib.SMTP(config.SMTP_SERVER, config.SMTP_PORT) as server:
            server.starttls()
            server.login(config.EMAIL_ADDRESS, config.EMAIL_APP_PASSWORD)
            server.sendmail(config.EMAIL_ADDRESS, recipient, msg.as_string())
        return True

    def update_dashboard(self, report_type: str, stats: dict, insights: dict, ai_commentary: str,
                          anomalies: list = None, context_summary: str = None, recommendations: list = None) -> str:
        payload = {
            "updated_at": datetime.datetime.now().isoformat(timespec="seconds"),
            "report_type": report_type,
            "prepared_for": config.PREPARED_FOR,
            "stats": stats,
            "insights": insights,
            "ai_commentary": ai_commentary,
            "anomalies": anomalies or [],
            "context_summary": context_summary,
            "recommendations": recommendations or [],
        }
        with open(config.DASHBOARD_STATE_PATH, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, default=str)
        return config.DASHBOARD_STATE_PATH

Writing agents/action_agent.py


In [ ]:
%%writefile orchestrator.py
import time
import threading

from agents.local_llm import LocalLLM
from agents.schema_agent import SchemaAgent
from agents.retrieval_agent import RetrievalAgent
from agents.analysis_agent import AnalysisAgent
from agents.action_agent import ActionAgent
import config


class Orchestrator:
    def __init__(self):
        self.llm = LocalLLM(config.LLM_MODEL_NAME)
        self.schema_agent = SchemaAgent(self.llm, config.COLUMN_MAP_CACHE_PATH)
        self.retrieval = RetrievalAgent(self.schema_agent)
        self.analysis = AnalysisAgent(self.llm)
        self.action = ActionAgent()

    def _retry(self, func, max_retries=None, backoff=None):
        max_retries = max_retries or config.MAX_RETRIES
        backoff = backoff or config.RETRY_BACKOFF_SECONDS
        last_exc = None
        for attempt in range(1, max_retries + 1):
            try:
                return func()
            except Exception as e:
                last_exc = e
                if attempt < max_retries:
                    time.sleep(backoff * attempt)
        raise last_exc

    def run(self, csv_path: str, report_type: str = "overall", send_email: bool = True,
            recipient: str = None, manual_overrides: dict = None) -> dict:
        log = {"steps": {}, "success": True, "report_type": report_type}

        t0 = time.time()
        try:
            data = self._retry(lambda: self.retrieval.retrieve_all(csv_path, manual_overrides=manual_overrides))
            log["steps"]["retrieval"] = {"ok": True, "seconds": round(time.time() - t0, 2)}
        except Exception as e:
            log["steps"]["retrieval"] = {"ok": False, "error": str(e)}
            log["success"] = False
            return log

        t0 = time.time()
        try:
            def _analyze():
                stats = self.analysis.compute_stats(data["sales_df"], data["columns"])
                anomalies = self.analysis.detect_anomalies(data["sales_df"], data["columns"])
                recommendations = self.analysis.generate_recommendations(report_type, stats, anomalies)
                insights = self.analysis.build_bilingual_summary(report_type, stats, data["usd_to_egp"], anomalies)
                ai_commentary = self.analysis.generate_ai_commentary(report_type, stats)
                sales_values = self.analysis.get_sales_series(data["sales_df"], data["columns"])
                return stats, anomalies, recommendations, insights, ai_commentary, sales_values

            stats, anomalies, recommendations, insights, ai_commentary, sales_values = self._retry(_analyze)
            log["steps"]["analysis"] = {"ok": True, "seconds": round(time.time() - t0, 2)}
        except Exception as e:
            log["steps"]["analysis"] = {"ok": False, "error": str(e)}
            log["success"] = False
            return log

        t0 = time.time()
        results = {}

        def _report():
            try:
                results["report_path"] = self.action.generate_report(
                    report_type, stats, insights, ai_commentary, anomalies,
                    data.get("context_summary"), recommendations, sales_values
                )
            except Exception as e:
                results["report_error"] = str(e)

        def _dashboard():
            try:
                results["dashboard_path"] = self.action.update_dashboard(
                    report_type, stats, insights, ai_commentary, anomalies,
                    data.get("context_summary"), recommendations
                )
            except Exception as e:
                results["dashboard_error"] = str(e)

        threads = [threading.Thread(target=_report), threading.Thread(target=_dashboard)]
        for t in threads:
            t.start()
        for t in threads:
            t.join()

        email_ok = False
        if send_email and "report_path" in results:
            try:
                email_ok = self._retry(lambda: self.action.send_email(report_type, results["report_path"], insights, recipient))
            except Exception as e:
                results["email_error"] = str(e)

        actions_ok = "report_path" in results and "dashboard_path" in results
        log["steps"]["actions"] = {"ok": actions_ok, "seconds": round(time.time() - t0, 2), "email_sent": email_ok}
        if "report_error" in results:
            log["steps"]["actions"]["report_error"] = results["report_error"]
        if "dashboard_error" in results:
            log["steps"]["actions"]["dashboard_error"] = results["dashboard_error"]
        if not actions_ok:
            log["success"] = False

        log["stats"] = stats
        log["anomalies"] = anomalies
        log["recommendations"] = recommendations
        log["insights"] = insights
        log["ai_commentary"] = ai_commentary
        log["context_summary"] = data.get("context_summary")
        log["report_path"] = results.get("report_path")
        log["dashboard_path"] = results.get("dashboard_path")
        log["columns_used"] = data.get("columns")
        return log

Writing orchestrator.py


In [ ]:
%%writefile evaluate.py
from orchestrator import Orchestrator
import config


def run_evaluation(csv_path: str = config.DEFAULT_CSV_PATH, report_type: str = "overall", runs: int = 3):
    orch = Orchestrator()
    results = [orch.run(csv_path, report_type=report_type, send_email=False) for _ in range(runs)]

    successes = sum(1 for r in results if r["success"])
    avg_times = {}
    for step in ["retrieval", "analysis", "actions"]:
        times = [r["steps"][step]["seconds"] for r in results if step in r["steps"] and r["steps"][step].get("ok")]
        if times:
            avg_times[step] = round(sum(times) / len(times), 2)

    summary = {"runs": runs, "success_rate": f"{successes}/{runs}", "avg_seconds_per_step": avg_times}
    return summary, results


if __name__ == "__main__":
    summary, _ = run_evaluation()
    print(summary)

Writing evaluate.py


In [ ]:
%%writefile app.py
import json
import os

import pandas as pd
import plotly.express as px
import streamlit as st

import config
from orchestrator import Orchestrator
from evaluate import run_evaluation

st.set_page_config(page_title="GTA V Multi-Agent Analytics", page_icon="🎮", layout="wide")

st.markdown("""
<style>
.main {background-color: #0e1117;}
h1, h2, h3 {font-family: 'Segoe UI', sans-serif;}
.stButton>button {border-radius: 8px; font-weight: 600;}
</style>
""", unsafe_allow_html=True)

st.title("🎮 GTA V Worldwide Sales — Multi-Agent AI System")
st.caption("Choose a report, run the pipeline, and it's emailed to the Marketing Team Lead automatically.")

csv_path = config.DEFAULT_CSV_PATH

@st.cache_resource
def get_orchestrator():
    return Orchestrator()

report_options = {v["title"]: k for k, v in config.REPORT_TYPES.items()}

with st.sidebar:
    st.header("Pipeline Controls")
    st.caption(f"Dataset path: `{csv_path}`")
    report_label = st.selectbox("Choose a report to generate", list(report_options.keys()))
    report_type = report_options[report_label]
    recipient = st.text_input("Send report to", value=config.RECIPIENT_EMAIL)
    st.caption(f"Prepared for: {config.PREPARED_FOR} — email is always sent on run.")
    run_btn = st.button("▶ Run Multi-Agent Pipeline", use_container_width=True)
    eval_btn = st.button("🧪 Run Evaluation (3x, no email)", use_container_width=True)

    st.divider()
    with st.expander("🔧 Column Mapping (auto-detected — correct if wrong)"):
        if os.path.exists(csv_path):
            df_preview = pd.read_csv(csv_path)
            df_preview.columns = [c.strip() for c in df_preview.columns]
            orch_preview = get_orchestrator()
            current_map = orch_preview.schema_agent.detect_columns(df_preview)
            st.caption("The Schema Agent classified these using the local model. Fix any mistake and save — it will be remembered for this dataset going forward.")
            all_cols = ["(none)"] + list(df_preview.columns)
            role_choices = {}
            for role in ["country", "platform", "sales", "players", "date"]:
                default = current_map.get(role, "(none)")
                idx = all_cols.index(default) if default in all_cols else 0
                choice = st.selectbox(f"{role.capitalize()} column", all_cols, index=idx, key=f"map_{role}")
                role_choices[role] = choice
            if st.button("💾 Save corrected mapping"):
                orch_preview.schema_agent.detect_columns(df_preview, manual_overrides=role_choices)
                st.success("Saved — this mapping will be reused automatically for this dataset from now on.")
        else:
            st.caption("CSV not found yet.")

tab_overview, tab_insights, tab_actions, tab_eval = st.tabs(
    ["📊 Data Overview", "🧠 AI Insights", "⚙️ Automated Actions", "✅ Evaluation"]
)

if run_btn:
    if not os.path.exists(csv_path):
        st.error(f"CSV not found at {csv_path}.")
    else:
        status = st.status(f"Running: {report_label}...", expanded=True)
        status.write("🔎 Retrieval Agent: loading CSV (using learned column mapping) + FX rate + background context...")
        orch = get_orchestrator()
        result = orch.run(csv_path, report_type=report_type, send_email=True, recipient=recipient)
        if result["success"]:
            status.write("🧠 Analysis Agent: stats + anomalies + bilingual summary + AI commentary done.")
            tail = ", email sent." if result["steps"]["actions"]["email_sent"] else ", email FAILED."
            status.write(f"⚙️ Action Agent: {report_label} generated" + tail)
            status.update(label="Pipeline complete ✅", state="complete")
            st.session_state["last_result"] = result
        else:
            status.update(label="Pipeline failed ❌", state="error")
            st.json(result)

if eval_btn:
    with st.spinner("Running pipeline 3 times for evaluation..."):
        summary, runs = run_evaluation(csv_path, report_type=report_type)
    st.session_state["eval_summary"] = summary

result = st.session_state.get("last_result")

with tab_overview:
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        st.dataframe(df.head(20), use_container_width=True)
        numeric_cols = df.select_dtypes("number").columns.tolist()
        if numeric_cols:
            col = st.selectbox("Chart a numeric column", numeric_cols)
            st.plotly_chart(px.histogram(df, x=col, nbins=30, title=f"Distribution of {col}"), use_container_width=True)
    else:
        st.warning(f"CSV not found at {csv_path}.")

with tab_insights:
    if result:
        st.caption(f"Report type: **{config.REPORT_TYPES[result['report_type']]['title']}** — Prepared for: {config.PREPARED_FOR}")
        if result.get("columns_used"):
            st.caption(f"Columns used: {result['columns_used']}")
        col_en, col_ar = st.columns(2)
        with col_en:
            st.subheader("Summary (English)")
            st.write(result["insights"].get("en", ""))
        with col_ar:
            st.subheader("الملخص (Arabic)")
            st.write(result["insights"].get("ar", ""))

        if result.get("ai_commentary"):
            st.subheader("🤖 AI Commentary")
            st.caption(result["ai_commentary"])

        st.subheader("✅ Recommendations")
        for r in result.get("recommendations", []):
            st.write("• " + r)

        if result.get("anomalies"):
            st.subheader("⚠️ Detected anomalies")
            st.table(result["anomalies"])

        st.subheader("Underlying stats")
        st.json(result["stats"])
    else:
        st.info("Run the pipeline to generate AI insights.")

with tab_actions:
    if result:
        c1, c2, c3 = st.columns(3)
        c1.metric("Report", "✅ Generated" if result.get("report_path") else "—")
        c2.metric("Dashboard", "✅ Updated" if result.get("dashboard_path") else "—")
        c3.metric("Email", "✅ Sent" if result["steps"]["actions"]["email_sent"] else "❌ Failed")

        if result.get("report_path") and os.path.exists(result["report_path"]):
            with open(result["report_path"], "rb") as f:
                st.download_button("⬇ Download PDF report", f, file_name=os.path.basename(result["report_path"]))

        if os.path.exists(config.DASHBOARD_STATE_PATH):
            with open(config.DASHBOARD_STATE_PATH) as f:
                st.json(json.load(f))
    else:
        st.info("Run the pipeline to trigger automated actions.")

with tab_eval:
    summary = st.session_state.get("eval_summary")
    if summary:
        st.subheader("Reliability & Efficiency")
        st.write(f"Success rate: **{summary['success_rate']}**")
        st.json(summary["avg_seconds_per_step"])
    else:
        st.info("Click 'Run Evaluation' in the sidebar to test reliability & speed across multiple runs.")

Writing app.py


In [ ]:
import os
if os.path.exists("column_map_cache.json"):
    os.remove("column_map_cache.json")

In [ ]:
!pkill -f streamlit
!pkill -f ngrok

In [ ]:
import subprocess, time
from pyngrok import ngrok, conf

conf.get_default().auth_token = os.environ["NGROK_AUTH_TOKEN"]
ngrok.kill()

log_file = open("streamlit_log.txt", "w")
proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=log_file, stderr=subprocess.STDOUT,
)
time.sleep(8)

public_url = ngrok.connect(8501)
print(f"Streamlit App URL: {public_url}")

Streamlit App URL: NgrokTunnel: "https://catcall-cultural-scribble.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
!python evaluate.py

config.json: 100% 1.40k/1.40k [00:00<00:00, 3.43MB/s]
tokenizer_config.json: 100% 2.54k/2.54k [00:00<00:00, 7.22MB/s]

spiece.model: downloading bytes:  64% 507k/792k [00:00<00:00, 703kB/s]
spiece.model: downloading bytes: 100% 516k/516k [00:00<00:00, 714kB/s, 50.8kB/s  ]
spiece.model: reconstructing file: 100% 792k/792k [00:00<00:00, 1.09MB/s, 78.0kB/s  ]
tokenizer.json: 100% 2.42M/2.42M [00:00<00:00, 42.9MB/s]
special_tokens_map.json: 100% 2.20k/2.20k [00:00<00:00, 7.01MB/s]
Loading weights: 100% 282/282 [00:00<00:00, 12975.77it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
generation_config.json: 100% 147/147 [00:00<00:00, 628kB/s]
{'runs': 3, 'success_rate': '3/3', 'avg_seconds_per_step': {'retrieval': 17.4, 'analysis': 3.91, 'actions': 1